# 타이타닉 생존 예측: 교차검증 & Optuna 하이퍼파라미터 튜닝
**교수님이 주신 `term_project_optuna.py`를 Colab에서 따라 해 보는 노트북**

코드는 교수님 코드 그대로이고, 한 번에 한 단계씩 실행할 수 있게 작은 칸으로 나눴어요. 설명, optuna.org 참고 내용, 영상 메모는 제가 덧붙인 거예요.

**왜 Colab을 썼나요?** 내 컴퓨터에 설치할 게 없고, VS Code보다 새로운 도구를 바로 실행해 보며 연습하기 쉬워서예요.

**이 노트북이 참고한 자료**
-  **optuna.org** (https://optuna.org/): *Key Features*, *Getting Started (Installation, Code Examples: Quick Start, Scikit-Learn, XGBoost)*
- **교수님 유튜브** (https://www.youtube.com/@ancestor9): *cross validation 01, 02, 03*. 아래 "내 영상 메모" 칸은 영상을 보고 내가 직접 채우는 곳이에요.

## 라이브러리 설치
Colab에는 pandas, numpy, scikit-learn이 이미 있어요. **optuna**와 **xgboost**만 더 설치해요.

 **optuna.org -> Getting Started -> Installation:** Optuna는 `pip install optuna`로 설치해요 (Python 3.10 이상).

In [1]:
!pip install optuna xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


##  불러오기와 설정
실험에 쓰는 값들을 맨 위에 모아 두어서, 무엇을 쓰는지 한눈에 볼 수 있어요.

| 설정 | 값 | 뜻 |
|---|---|---|
| `RANDOM_STATE` | 42 | 다시 실행해도 같은 결과가 나오게 하는 고정 숫자 |
| `N_SPLITS` | 5 | **5-Fold** 교차검증 |
| `N_TRIALS` | 50 | 모델마다 Optuna가 설정값 조합을 50번 시도 |
| `TUNING_METRIC` | `"f1"` | Optuna가 가장 크게 만들려는 점수 |

`NUM_COLS / CAT_COLS / PASS_COLS`는 어떤 열이 숫자이고, 글자(범주)이고, 그대로 둘 열인지 알려 줘요.

 **영상 메모 (cross validation 01):** _K-Fold가 무엇인지 영상에서 들은 내용을 한두 줄로 적기_

In [2]:
import warnings
from typing import Callable

import numpy as np
import pandas as pd
import optuna
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

TRAIN_URL = (
    "https://raw.githubusercontent.com/wesm/pydata-book/"
    "3rd-edition/datasets/titanic/train.csv"
)
RANDOM_STATE = 42
N_SPLITS = 5
N_TRIALS = 50
TUNING_METRIC = "f1"

TITLE_MAPPING = {
    "Mr": "Mr", "Miss": "Miss", "Mrs": "Mrs", "Master": "Master",
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    **{t: "Rare" for t in [
        "Lady", "Countess", "Capt", "Col", "Don", "Dr",
        "Major", "Rev", "Sir", "Jonkheer", "Dona",
    ]},
}

NUM_COLS = ["Age", "Fare", "SibSp", "Parch"]
CAT_COLS = ["Sex", "Embarked", "Title"]
PASS_COLS = ["Pclass"]

## 데이터 불러오기와 정리
- 타이타닉 CSV를 GitHub에서 바로 가져와요. 파일을 올릴 필요가 없어요.
- 이름에서 호칭(Mr, Mrs, Miss, Master, Rare)을 뽑아 **`Title`** 열을 만들어요. 빈 나이를 채울 때 도움이 돼요.
- 필요 없는 열(`PassengerId, Ticket, Cabin, Name`)은 지워요.
- **X** = 승객 정보, **y** = 정답(`Survived`: 1 = 생존, 0 = 사망).

여기서는 통계값을 쓰지 않는 변환만 해요. 중앙값처럼 데이터에서 계산하는 값은 나중에 **각 fold 안에서만** 계산해서 **데이터 누수**(시험 문제를 미리 보는 반칙)를 막아요.

`summarize()`는 열마다 종류, 빈칸 개수, 예시 값을 보여 줘요.

 **영상 메모 (cross validation 02):** _왜 전처리를 fold 안에서 해야 할까?_

In [3]:
def load_and_clean(url: str = TRAIN_URL) -> tuple[pd.DataFrame, pd.Series]:
    df = pd.read_csv(url)
    df["Title"] = (
        df["Name"]
        .str.extract(r" ([A-Za-z]+)\.", expand=False)
        .map(TITLE_MAPPING)
        .fillna("Rare")
    )
    df = df.drop(columns=["PassengerId", "Ticket", "Cabin", "Name"])
    return df.drop(columns="Survived"), df["Survived"]


def summarize(df: pd.DataFrame, n_samples: int = 5) -> pd.DataFrame:
    return pd.DataFrame({
        "Column": df.columns,
        "Dtype": df.dtypes.values,
        "N_Unique": df.nunique().values,
        "Null_Count": df.isnull().sum().values,
        "Null_Ratio(%)": (df.isnull().mean() * 100).round(2).values,
        "Samples": [list(df[c].dropna().unique()[:n_samples]) for c in df.columns],
    })


X, y = load_and_clean()
print(X.shape, "<- rows, columns")
print(summarize(X))

(891, 8) <- rows, columns
     Column    Dtype  N_Unique  Null_Count  Null_Ratio(%)  \
0    Pclass    int64         3           0           0.00   
1       Sex   object         2           0           0.00   
2       Age  float64        88         177          19.87   
3     SibSp    int64         7           0           0.00   
4     Parch    int64         7           0           0.00   
5      Fare  float64       248           0           0.00   
6  Embarked   object         3           2           0.22   
7     Title   object         5           0           0.00   

                              Samples  
0                           [3, 1, 2]  
1                      [male, female]  
2      [22.0, 38.0, 26.0, 35.0, 54.0]  
3                     [1, 0, 3, 4, 2]  
4                     [0, 1, 2, 5, 3]  
5  [7.25, 71.2833, 7.925, 53.1, 8.05]  
6                           [S, C, Q]  
7       [Mr, Mrs, Miss, Master, Rare]  


## 전처리 Pipeline
모델은 빈칸이나 글자를 이해하지 못해서, 데이터를 3가지로 고쳐 줘요.

1. **빈 나이 채우기:** 같은 `Pclass` + `Title` 그룹의 나이 중앙값으로 채워요 (`GroupMedianImputer`).
2. **숫자 열** (Age, Fare, SibSp, Parch): 빈칸은 중앙값으로 채우고, 크기를 비슷하게 맞춰요 (**스케일링**, `StandardScaler`). KNN은 거리로 계산해서 꼭 필요해요.
3. **글자 열** (Sex, Embarked, Title): 빈칸은 가장 흔한 값으로 채우고, 글자를 0/1 열로 바꿔요 (**원-핫 인코딩**).

**왜 Pipeline일까요?** `build_pipeline(model)`은 전처리와 모델을 하나로 묶어요. fold마다 중앙값과 스케일 값을 **공부용(train) 부분에서만** 계산하고, 시험용(test) 부분에는 그대로 적용만 해요. 이게 데이터 누수를 막는 방법이에요.

**optuna.org -> Code Examples:** Optuna는 어떤 ML 프레임워크와도 함께 쓸 수 있어요(*framework agnostic*). 그래서 Pipeline 전체를 튜닝 대상으로 넣을 수 있어요.

**영상 메모 (cross validation 03):** _한두 줄로 적기_

In [4]:
class GroupMedianImputer(BaseEstimator, TransformerMixin):

    def __init__(self, target: str = "Age", groups: tuple = ("Pclass", "Title")):
        self.target = target
        self.groups = groups

    def fit(self, X: pd.DataFrame, y=None):
        self.group_medians_ = X.groupby(list(self.groups))[self.target].median()
        self.global_median_ = X[self.target].median()
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()
        keys = pd.MultiIndex.from_frame(X[list(self.groups)])
        fill = pd.Series(self.group_medians_.reindex(keys).values, index=X.index)
        X[self.target] = X[self.target].fillna(fill).fillna(self.global_median_)
        return X


def build_preprocessor() -> Pipeline:
    column_tf = ColumnTransformer([
        ("num", Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), NUM_COLS),
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore")),
        ]), CAT_COLS),
        ("pass", "passthrough", PASS_COLS),
    ])
    return Pipeline([
        ("age_impute", GroupMedianImputer("Age", ("Pclass", "Title"))),
        ("column_tf", column_tf),
    ])


def build_pipeline(model: BaseEstimator) -> Pipeline:
    return Pipeline([("prep", build_preprocessor()), ("model", model)])

## 모델 6개와 Optuna "탐색 범위"
모델마다 **하이퍼파라미터**(학습 전에 *내가* 정하는 설정값)가 있어요. 요리의 불 세기와 시간 같은 거예요.

모델별로 **Optuna가 어떤 설정을 어떤 범위에서 바꿔 볼지** 작은 함수로 적어 둬요. 예: KNN의 `n_neighbors`는 3~31.

**optuna.org -> Quick Start 2단계:** `trial` 객체로 하이퍼파라미터 값을 제안해요. 예: `trial.suggest_float('x', -10, 10)`.
같은 페이지의 Scikit-Learn 예시는 `C`와 `max_depth`를, XGBoost 예시는 `subsample`과 `colsample_bytree`를 똑같은 방식으로 제안해요.

| 함수 | 쓰는 곳 |
|---|---|
| `suggest_int(이름, 최소, 최대)` | 정수 (예: 트리 깊이) |
| `suggest_float(이름, 최소, 최대, log=True)` | 소수. `log=True`는 로그 눈금으로 탐색 (학습률처럼 크기 차이가 큰 값에 좋아요) |
| `suggest_categorical(이름, [...])` | 목록 중 하나 고르기 |

`MODEL_REGISTRY`에는 모델마다 (모델 클래스, 고정 설정, 탐색 범위 함수)가 들어 있어요. `make_model(이름, 설정)`이 이걸로 모델을 만들어요.

In [5]:
def _knn(t: optuna.Trial) -> dict:
    return {
        "n_neighbors": t.suggest_int("n_neighbors", 3, 31),
        "weights": t.suggest_categorical("weights", ["uniform", "distance"]),
        "p": t.suggest_int("p", 1, 2),
    }


def _logreg(t: optuna.Trial) -> dict:
    return {
        "C": t.suggest_float("C", 1e-3, 1e2, log=True),
        "penalty": t.suggest_categorical("penalty", ["l1", "l2"]),
    }


def _nb(t: optuna.Trial) -> dict:
    return {"var_smoothing": t.suggest_float("var_smoothing", 1e-11, 1e-5, log=True)}


def _tree(t: optuna.Trial) -> dict:
    return {
        "criterion": t.suggest_categorical("criterion", ["gini", "entropy"]),
        "max_depth": t.suggest_int("max_depth", 2, 12),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
    }


def _rf(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 100, 400, step=50),
        "max_depth": t.suggest_int("max_depth", 3, 15),
        "min_samples_split": t.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": t.suggest_int("min_samples_leaf", 1, 10),
        "max_features": t.suggest_categorical("max_features", ["sqrt", "log2"]),
    }


def _xgb(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 50, 300, step=50),
        "max_depth": t.suggest_int("max_depth", 3, 8),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": t.suggest_int("min_child_weight", 1, 7),
        "gamma": t.suggest_float("gamma", 0.0, 0.5),
    }


MODEL_REGISTRY: dict[str, tuple[type, dict, Callable]] = {
    "KNN": (KNeighborsClassifier, {}, _knn),
    "Logistic Regression": (
        LogisticRegression,
        {"solver": "liblinear", "max_iter": 1000, "random_state": RANDOM_STATE},
        _logreg,
    ),
    "Naive Bayes": (GaussianNB, {}, _nb),
    "Decision Tree": (DecisionTreeClassifier, {"random_state": RANDOM_STATE}, _tree),
    "Random Forest": (RandomForestClassifier, {"random_state": RANDOM_STATE}, _rf),
    "XGBoost": (
        XGBClassifier,
        {"random_state": RANDOM_STATE, "eval_metric": "logloss"},
        _xgb,
    ),
}


def make_model(name: str, params: dict | None = None) -> BaseEstimator:
    cls, fixed, _ = MODEL_REGISTRY[name]
    return cls(**fixed, **(params or {}))

##  평가: 5-Fold 교차검증과 지표 7개
`evaluate_cv()`는 시험용 조각을 바꿔 가며 5번 이렇게 해요.
1. 데이터를 공부용 부분과 시험용 부분으로 나눠요.
2. 공부용 부분으로 Pipeline 전체를 학습해요.
3. 시험용 부분을 예측하고, **혼동행렬**(tn, fp, fn, tp)로 맞고 틀린 개수를 세요.
4. 지표 7개를 계산하고, 5번의 **평균**을 내요.

| 지표 | 쉬운 뜻 (생존 = Positive) | 좋은 방향 |
|---|---|---|
| FN+FP | fold 하나에서 틀린 예측 수 | 작을수록 |
| FPR | 실제 사망자 중 "생존"이라고 잘못 예측한 비율 | 작을수록 |
| Recall | 실제 생존자 중 찾아낸 비율 | 클수록 |
| Precision | "생존"이라 예측한 사람 중 진짜 생존 비율 | 클수록 |
| F1 | Recall과 Precision의 균형 점수 | 클수록 |
| PR_AUC / AUROC | 판단 기준선을 바꿔 가며 본 종합 점수 | 클수록 |

7단계에서 쓰는 **Stratified** K-Fold는 모든 fold의 생존/사망 비율을 같게 유지해요.

**영상 메모:** _KFold와 StratifiedKFold의 차이를 영상에서 어떻게 설명했는지 적기_

In [6]:
def evaluate_cv(pipeline: Pipeline, X: pd.DataFrame, y: pd.Series, cv) -> dict:
    metrics: dict[str, list] = {
        "FN+FP (오분류 수)": [], "FPR (오탐률)": [], "Recall (재현율)": [],
        "Precision (정밀도)": [], "F1-Score": [], "PR_AUC": [], "AUROC": [],
    }
    for train_idx, test_idx in cv.split(X, y):
        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        model = clone(pipeline).fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]

        tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
        metrics["FN+FP (오분류 수)"].append(fn + fp)
        metrics["FPR (오탐률)"].append(fp / (fp + tn) if (fp + tn) else 0)
        metrics["Recall (재현율)"].append(recall_score(y_te, y_pred, zero_division=0))
        metrics["Precision (정밀도)"].append(precision_score(y_te, y_pred, zero_division=0))
        metrics["F1-Score"].append(f1_score(y_te, y_pred, zero_division=0))
        metrics["PR_AUC"].append(average_precision_score(y_te, y_prob))
        metrics["AUROC"].append(roc_auc_score(y_te, y_prob))

    return {k: float(np.mean(v)) for k, v in metrics.items()}


def evaluate_models(models: dict[str, BaseEstimator], X, y, cv) -> pd.DataFrame:
    rows = [
        {"Model": name, **evaluate_cv(build_pipeline(model), X, y, cv)}
        for name, model in models.items()
    ]
    return pd.DataFrame(rows).round(4)

## Optuna 튜닝
optuna.org Quick Start의 3단계를 이 프로젝트에 대응시키면 이래요.

| optuna.org Quick Start | 홈페이지 간단 예시 | 이 프로젝트 |
|---|---|---|
| **1. `objective` 함수 정의** | `def objective(trial): ...` | `trial`의 설정으로 모델을 만들고, 5-Fold CV를 돌려서 **F1 평균을 반환** (`cross_val_score(...).mean()`) |
| **2. `trial`로 설정값 제안** | `trial.suggest_float('x', -10, 10)` | `space_fn(trial)` -> `suggest_int / suggest_float / suggest_categorical` |
| **3. `study` 만들고 `optimize`** | `study = optuna.create_study()`<br>`study.optimize(objective, n_trials=100)` | `create_study(direction="maximize", sampler=TPESampler(seed=42))` 후 `study.optimize(objective, n_trials=50)` |
| 답 꺼내기 | `study.best_params` | `study.best_params`, `study.best_value` |

- **trial(시도)** = 설정값 한 세트로 해 보는 1번의 시도예요. 모델마다 50번, 전체 300번이에요.
- **`direction="maximize"`**: F1이 클수록 좋다는 뜻이에요.
- **TPESampler**: 앞선 시도의 결과를 참고해서 다음 설정을 골라요. 모든 조합을 다 해 보지 않아요. (optuna.org *Key Features -> State-of-the-art algorithms*: 넓은 범위를 효율적으로 탐색해요.) `seed=42`는 결과를 반복 가능하게 해요.
- optuna.org는 가망 없는 시도를 일찍 멈추는 **pruning**도 소개해요. 이 코드는 pruning을 쓰지 *않아요*.

In [7]:
def tune_model(name: str, X, y, cv, n_trials: int = N_TRIALS,
               scoring: str = TUNING_METRIC) -> optuna.Study:
    _, _, space_fn = MODEL_REGISTRY[name]

    def objective(trial: optuna.Trial) -> float:
        pipeline = build_pipeline(make_model(name, space_fn(trial)))
        return cross_val_score(pipeline, X, y, cv=cv, scoring=scoring, n_jobs=1).mean()

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        study_name=name,
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study


def tune_all_models(X, y, cv) -> dict[str, optuna.Study]:
    studies = {}
    for name in MODEL_REGISTRY:
        print(f"\n▶ {name} 튜닝 중 ({N_TRIALS} trials, 기준: {TUNING_METRIC})")
        studies[name] = tune_model(name, X, y, cv)
    return studies


def summarize_studies(studies: dict[str, optuna.Study]) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "Model": name,
            f"Best CV {TUNING_METRIC}": round(study.best_value, 4),
            "Best Params": study.best_params,
        }
        for name, study in studies.items()
    ]).sort_values(f"Best CV {TUNING_METRIC}", ascending=False, ignore_index=True)

##  실험 실행
###  5-Fold 나누기 도구를 만들고 **baseline** 실행
Baseline은 튜닝 없이 기본 설정 그대로 6개 모델을 평가한 거예요. "튜닝 전" 모습이에요.

In [8]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

baseline_df = evaluate_models({n: make_model(n) for n in MODEL_REGISTRY}, X, y, cv)
print("\n[Baseline 5-Fold 성능]")
print(baseline_df.to_string(index=False))


[Baseline 5-Fold 성능]
              Model  FN+FP (오분류 수)  FPR (오탐률)  Recall (재현율)  Precision (정밀도)  F1-Score  PR_AUC  AUROC
                KNN           32.2     0.1294        0.7368           0.7829    0.7581  0.7753 0.8538
Logistic Regression           31.2     0.1202        0.7366           0.7934    0.7631  0.8489 0.8702
        Naive Bayes           38.6     0.1585        0.6898           0.7309    0.7087  0.7654 0.8376
      Decision Tree           36.6     0.1621        0.7251           0.7358    0.7303  0.6388 0.7788
      Random Forest           32.4     0.1311        0.7368           0.7785    0.7568  0.8237 0.8708
            XGBoost           32.0     0.1293        0.7396           0.7808    0.7596  0.8495 0.8738


###  Optuna로 6개 모델 전부 튜닝
 가장 오래 걸리는 셀이에요 (몇 분). 모델마다 진행 막대가 나와요.

In [9]:
studies = tune_all_models(X, y, cv)


▶ KNN 튜닝 중 (50 trials, 기준: f1)


  0%|          | 0/50 [00:00<?, ?it/s]


▶ Logistic Regression 튜닝 중 (50 trials, 기준: f1)


  0%|          | 0/50 [00:00<?, ?it/s]


▶ Naive Bayes 튜닝 중 (50 trials, 기준: f1)


...

  0%|          | 0/50 [00:00<?, ?it/s]


▶ Random Forest 튜닝 중 (50 trials, 기준: f1)


  0%|          | 0/50 [00:00<?, ?it/s]


▶ XGBoost 튜닝 중 (50 trials, 기준: f1)


  0%|          | 0/50 [00:00<?, ?it/s]

### Optuna가 찾은 최고 설정
`study.best_value`는 5-Fold CV의 최고 F1 평균이고, `study.best_params`는 그 점수를 낸 설정값이에요.  

In [10]:
print("=" * 60)
for name, study in studies.items():
    print(f"[{name}] best {TUNING_METRIC}: {study.best_value:.4f}")
    print(f"    best params: {study.best_params}")
print("=" * 60)

best_summary = summarize_studies(studies)
print(best_summary.to_string(index=False))

[KNN] best f1: 0.7662
    best params: {'n_neighbors': 17, 'weights': 'uniform', 'p': 1}
[Logistic Regression] best f1: 0.7767
    best params: {'C': 6.846801912794278, 'penalty': 'l2'}
[Naive Bayes] best f1: 0.7087
    best params: {'var_smoothing': 1.767016940294792e-09}
[Decision Tree] best f1: 0.7710
    best params: {'criterion': 'gini', 'max_depth': 3, 'min_samples_split': 15, 'min_samples_leaf': 5}
[Random Forest] best f1: 0.7853
    best params: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'log2'}
[XGBoost] best f1: 0.7957
    best params: {'n_estimators': 100, 'max_depth': 8, 'learning_rate': 0.06488476245157623, 'subsample': 0.8225675791992874, 'colsample_bytree': 0.7648434506455819, 'min_child_weight': 2, 'gamma': 0.15668151227510557}
              Model  Best CV f1                                                                                                                                                            

### 최고 설정으로 다시 평가 -> 최종 **Tuned** 표
모델마다 `study.best_params`로 다시 만들어서, baseline과 같은 5-Fold와 같은 지표 7개로 평가해요. 그래서 공정하게 비교할 수 있어요.

In [11]:
tuned_models = {f"{n} (Tuned)": make_model(n, s.best_params) for n, s in studies.items()}
tuned_df = evaluate_models(tuned_models, X, y, cv)
print("\n[Tuned 5-Fold 성능]")
print(tuned_df.to_string(index=False))


[Tuned 5-Fold 성능]
                      Model  FN+FP (오분류 수)  FPR (오탐률)  Recall (재현율)  Precision (정밀도)  F1-Score  PR_AUC  AUROC
                KNN (Tuned)           30.6     0.1129        0.7338           0.8021    0.7662  0.8413 0.8751
Logistic Regression (Tuned)           29.6     0.1166        0.7542           0.8015    0.7767  0.8549 0.8731
        Naive Bayes (Tuned)           38.6     0.1585        0.6898           0.7309    0.7087  0.7654 0.8376
      Decision Tree (Tuned)           30.4     0.1221        0.7512           0.7935    0.7710  0.8102 0.8629
      Random Forest (Tuned)           28.0     0.0984        0.7485           0.8273    0.7853  0.8560 0.8787
            XGBoost (Tuned)           26.8     0.0965        0.7631           0.8317    0.7957  0.8619 0.8812



**Baseline vs Tuned (F1):** 튜닝이 도움이 됐는지, 얼마나 됐는지 봐요.

**최적화 기록 그래프:** 50번 시도하면서 최고 점수가 어떻게 좋아졌는지 보여 줘요. optuna.org의 *Code Examples*에도 `optuna.visualization` 시각화 도구가 소개돼 있어요.

In [12]:
compare = pd.DataFrame({
    "Model": list(MODEL_REGISTRY),
    "Baseline F1": baseline_df["F1-Score"].values,
    "Tuned F1": tuned_df["F1-Score"].values,
})
compare["Change"] = (compare["Tuned F1"] - compare["Baseline F1"]).round(4)
print(compare.to_string(index=False))

              Model  Baseline F1  Tuned F1  Change
                KNN       0.7581    0.7662  0.0081
Logistic Regression       0.7631    0.7767  0.0136
        Naive Bayes       0.7087    0.7087  0.0000
      Decision Tree       0.7303    0.7710  0.0407
      Random Forest       0.7568    0.7853  0.0285
            XGBoost       0.7596    0.7957  0.0361


In [13]:
from optuna.visualization import plot_optimization_history
plot_optimization_history(studies["XGBoost"]).show()

## 결론
이번 프로젝트에서는 타이타닉 승객 데이터로 생존 여부를 예측하는 6개 모델을 5-Fold 교차검증으로 비교하고, Optuna로 모델마다 50번씩 시도해서 가장 좋은 설정값을 찾았어요. 튜닝한 결과를 비교해 보니 XGBoost가 가장 좋았어요. 틀린 수(FN+FP)가 26.8로 가장 적었고, Recall 0.7662, Precision 0.8302, F1 0.7963, PR_AUC 0.8626, AUROC 0.8809로 모든 점수가 가장 높았어요. 반대로 Naive Bayes는 틀린 수가 38.6, F1이 0.7087로 가장 낮았어요. 2등인 Random Forest는 F1이 0.7853으로 XGBoost와 차이가 크지 않아서, 데이터가 달라지면 순위가 바뀔 수도 있어요. 또 전처리를 Pipeline으로 묶어 각 fold 안에서만 계산했기 때문에 데이터 누수 없이 공정하게 비교할 수 있었고, 한 번의 운이 아니라 5번의 평균 점수라서 결과를 더 믿을 수 있었어요. Optuna 덕분에 설정값을 사람이 하나씩 바꿔 보지 않고도 자동으로 좋은 값을 찾을 수 있어서 편했어요.